### Задание 1 (8 баллов + 0.5)

Скачайте вот этот датасет - ("MERA-evaluation/MERA", "multiq"). Так как обучать ничего не придется возьмите train split. Возьмите оттуда 25 примеров (случайных, а не первых).

Преобразуйте колонку instruction, заменив переменные вроде `{support_text}` на реальные значения из других колонок, чтобы получился готовый промпт (инпут для модели).

In [ ]:
!pip install pymorphy3

In [2]:
from datasets import load_dataset
import re
import string
from collections import Counter
from tqdm import tqdm

import pymorphy3
morph = pymorphy3.MorphAnalyzer()

c:\Users\katyk\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ds = load_dataset("MERA-evaluation/MERA", "multiq", split="train")
mera = ds.shuffle(seed=42).select(range(25)) # сэмпл из 25 примеров

Generating test split: 100%|██████████| 900/900 [00:00<00:00, 27294.23 examples/s]


In [5]:
def build_prompt(row):
    inputs = row["inputs"]
    prompt = row["instruction"].replace("{support_text}", inputs["support_text"]).replace("{text}", inputs["text"]).replace("{question}", inputs["question"])
    return {"instruction": prompt}

mera = mera.map(build_prompt)

Map:   0%|          | 0/25 [00:00<?, ? examples/s]

Map: 100%|██████████| 25/25 [00:00<00:00, 289.04 examples/s]


#### Метрика (2 балла)

Напишите две метрики для оценки правильных ответов: 1) exact match и 2) F1 по леммам. Для первой просто добавьте базовую предобработку вроде удаления пунктуации и регистра и просто сопоставьте строки между собой. Для второй к базовой предобработке добавьте токенизацию и лемматизацию, а поверх лемм рассчитайте F1 метрику (подсказка - precision можно рассчитать посчитав какой процент лемм в предсказанном ответе совпадает с правильным ответом).

Рассчитайте метрики на простом бейзлайне (например, можете попробовать взять самый частотный биграмм из промпта в качестве ответа). Можно получить +0.5 балла за неочевидный ненулевой бейзлайн.

In [6]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Error loading punkt: <urlopen error [Errno 2] No such file
[nltk_data]     or directory>
[nltk_data] Error loading punkt_tab: <urlopen error [Errno 2] No such
[nltk_data]     file or directory>


False

In [7]:
def preprocessing(answer, mode):  # basic / pro
    answer = re.sub(r"[^\w\s]", "", answer.lower())
    answer = " ".join(answer.split())
    if mode == "basic":
        return answer
    if mode == "pro":
        a_tokens = word_tokenize(answer, language="russian") # для F1
        return [morph.parse(t)[0].normal_form for t in a_tokens]

В MERA иногда эталоны разделены точкой с запятой. Учтем это для правильного подсчёта метрик.

In [8]:
def split_golds(gold):
    return [g.strip() for g in gold.split(";")]

def exact_match(pred, gold):
    p = preprocessing(pred, "basic")
    return max(float(p == preprocessing(g, "basic")) for g in split_golds(gold))

def f1(pred, gold):
    p = preprocessing(pred, "pro")
    best = 0.0
    for g in split_golds(gold):
        r = preprocessing(g, "pro")
        common = sum((Counter(p) & Counter(r)).values())
        if common:
            pr, rc = common / len(p), common / len(r)
            best = max(best, 2 * pr * rc / (pr + rc))
    return best

Этот бэйзлайн ищет слова, которые есть в тексте, но не в вопросе, и выбирает самое длинное из них. И дает нулевые результаты((

In [10]:
def baseline(example):
    question_words = set(preprocessing(example["inputs"]["question"], "basic"))
    candidates = [w for w in re.findall(r"\w+", example["inputs"]["support_text"].lower()) if w not in question_words]
    return max(candidates, key=len) if candidates else ""

b_em = [exact_match(baseline(x), x["outputs"]) for x in mera]
b_f1 = [f1(baseline(x), x["outputs"]) for x in mera]

print("exact match:", sum(b_em) / len(b_em))
print("F1:", sum(b_f1) / len(b_f1))

exact match: 0.0
F1: 0.0


#### Эксперимент 1 (2 балла)
Возьмите какую-нибудь не очень большую модель с huggingface (например, `Qwen/Qwen3.5-0.8B`, но можно и любую другую, которая умеет отвечать на инструкции и поддерживает thinking). Преобразуйте промпты из датасета в формат `messages = [{"role": "user", "content": prompt}]` с помощью чат темплейта этой модели.
Сгенерируйте ответы модели на полученных данных (25 примеров). На этом шаге не используйте thinking. Поставьте max_new_tokens на не очень большое значения, чтобы генерация не занимала слишком много времени.
Оцените качество ответов с помощью двух метрик.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()

In [27]:
def generate(prompt, system_prompt=None):
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [16]:
predictions = [generate(x["instruction"]) for x in tqdm(mera)]

  0%|          | 0/25 [00:00<?, ?it/s][transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
100%|██████████| 25/25 [00:38<00:00,  1.

In [24]:
print("exact match:", sum(exact_match(p, x["outputs"]) for p, x in zip(predictions, mera)) / len(mera))
print("F1:", sum(f1(p, x["outputs"]) for p, x in zip(predictions, mera)) / len(mera))

exact match: 0.08
F1: 0.256


In [41]:
for p, x in zip(predictions, mera):
    print("вопрос:", x["inputs"]["question"])
    print("предсказание:", p)
    print("gold:", x["outputs"])
    print("exact match:", exact_match(p, x["outputs"]), ", F1:", round(f1(p, x["outputs"]), 2))
    print()

вопрос: В какую реку впадает река, притоком которой является Мурдюг?
предсказание: Тумана
gold: Каму;Камы
exact match: 0.0 , F1: 0.0


вопрос: Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?
предсказание: Глава государства, ставшего местом смерти Романа Николаевича Филипова, — это **Сирия**.

**Пояснение:**
В тексте первого предложения указано, что Роман Николаевич
gold: президентов
exact match: 0.0 , F1: 0.0


вопрос: Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?
предсказание: 4506 м
gold: Белуха;гора Белуха
exact match: 0.0 , F1: 0.0


вопрос: В какую реку впадает река, притоком которой является Суходревка?
предсказание: Суходревка впадает в реку Суходрев.
gold: Шаня
exact match: 0.0 , F1: 0.0


вопрос: Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?
предсказание: Ереванский городский совет
gold: ереванским городским советом
exact match: 0.0 , F1: 0.67


вопрос: Ка

Выдает много длинных предложений / пояснения. 

#### Эксперимент 2 (1 балла)
К первому эксперименту добавьте какой-то system prompt, который по вашему должен помочь модели справится с результатом. Можете проанализировать ошибки из первого задания и написать инструкции на основе этого.
Сгенерируйте ответы с точно такими же параметрами как и в эксперименте 1.
Оцените качество ответов с помощью двух метрик.

In [35]:
system_prompt = ("""Отвечай только текстом, без markdown, звёздочек, жирного шрифта, заголовков. Не пиши пояснений, не повторяй вопрос.
    Ответ — одно слово или имя собственное, без полного предложения. Назови ответ из текста в той форме, в какой он там написан. """)

In [37]:
predictions2 = [generate(x["instruction"], system_prompt=system_prompt) for x in tqdm(mera)]

100%|██████████| 25/25 [00:23<00:00,  1.05it/s]


In [38]:
print("exact match:", sum(exact_match(p, x["outputs"]) for p, x in zip(predictions2, mera)) / len(mera))
print("F1:", sum(f1(p, x["outputs"]) for p, x in zip(predictions2, mera)) / len(mera))

exact match: 0.08
F1: 0.26266666666666666


In [40]:
for p, x in zip(predictions2, mera):
    print("вопрос:", x["inputs"]["question"])
    print("предсказание:", p)
    print("gold:", x["outputs"])
    print("exact match:", exact_match(p, x["outputs"]), ", F1:", round(f1(p, x["outputs"]), 2))
    print()

вопрос: В какую реку впадает река, притоком которой является Мурдюг?
предсказание: Вятка
gold: Каму;Камы
exact match: 0.0 , F1: 0.0


вопрос: Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?
предсказание: Гвардии майор
gold: президентов
exact match: 0.0 , F1: 0.0


вопрос: Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?
предсказание: Белуха
gold: Белуха;гора Белуха
exact match: 1.0 , F1: 1.0


вопрос: В какую реку впадает река, притоком которой является Суходревка?
предсказание: Суходревка
gold: Шаня
exact match: 0.0 , F1: 0.0


вопрос: Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?
предсказание: Ереванский городский совет
gold: ереванским городским советом
exact match: 0.0 , F1: 0.67


вопрос: Как называется река, питающаяся от реки, притоком которой является Ворговаш?
предсказание: Ворговаш
gold: Сабля
exact match: 0.0 , F1: 0.0


вопрос: В какой лиге играет команда,





#### Эсперимент 3 (3 балла)
На этом шаге включите thinking в модели и сгенерируйте предсказания (system prompt вы можете использовать по вашему желанию).
На этом шаге у вас будет несколько сложностей, которые нужно решить:
1) генерацию модели нельзя использовать напрямую, сначала нужно отделить thinking блок от самого ответа. Добавьте пост обработку
2) thinking может занять достаточно много токенов и его нужно ограничить, но при этом вам нужно, чтобы модель выдала и сам ответ, а не только незаконченные размышлений. Выберите max_new_tokens с рассчетом на (thinking_tokens + answer_tokens). Сначала запустите первую генерацию с thinking_tokens, затем проверьте, что в ответе содержится </think> тэг, который означает, что модель закончила думать (еще удобно использовать `stop_strings` в huggingface generation параметрах, чтобы заканчивать как только появился нужный токен). И если тэг есть, то просто запустите вторую генерацию с max_new_tokens (как продолжение уже накопившегося) или же поставьте </think> тэг вручную (обрубив размышление) и запустите вторую генерацию c answer_tokens.


Немного изменяю системный промпт: пишу про размышления (не уверена, что это поможет, но попробовать можно)

In [73]:
system_prompt = ("""Отвечай только текстом, без markdown, звёздочек, жирного шрифта, заголовков. Не пиши пояснений, не повторяй вопрос.
    Ответ — одно слово или имя собственное, без полного предложения. Назови ответ из текста в той форме, в какой он там написан. Думай кратко, не более 3-4 предложений, затем сразу давай ответ.""")

In [ ]:
thinking_tokens = 512
answer_tokens = 40

think_end_id = tokenizer.convert_tokens_to_ids("</think>")
assert tokenizer.decode([think_end_id], skip_special_tokens=False) == "</think>" # проверка что всё ок, и </think> не бьется на несколько токенов и существует в словаре

In [76]:
base_eos = model.generation_config.eos_token_id
base_eos = base_eos if isinstance(base_eos, list) else [base_eos]
stop_ids = base_eos + [think_end_id]

# если рассуждение не уложилось в бюджет токенов обрубаем его этой фразой
FORCE_END = "\n\nНужно дать ответ прямо сейчас.\n</think>\n\nОтвет:"

In [77]:
# постобработка
def strip_thinking(text):
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return text.replace("<think>", "").strip()

In [ ]:
def generate_with_thinking(prompt, system_prompt=system_prompt):
    messages = [{"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        # рассуждение
        out = model.generate(
            **inputs,
            max_new_tokens=thinking_tokens,
            do_sample=False,
            eos_token_id=stop_ids,
        )
        last = out[0, -1].item()

        # модель сама закончила и думать и отвечать (выдала EOS)
        if last in base_eos:
            decoded = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True)
            return strip_thinking(decoded), True
        finished_thinking = last == think_end_id

        # рассуждение обрезано по длине, закрываем блок вручную
        if not finished_thinking:
            end = tokenizer(FORCE_END, add_special_tokens=False, return_tensors="pt")["input_ids"].to(model.device)
            out = torch.cat([out, end], dim=1)

        # ответ
        ans = model.generate(
            input_ids=out,
            attention_mask=torch.ones_like(out),
            max_new_tokens=answer_tokens,
            do_sample=False,
        )

    answer = tokenizer.decode(ans[0, out.shape[1]:], skip_special_tokens=True)
    return strip_thinking(answer), finished_thinking


Проверка на трёх примерах:

In [79]:
sample = mera.select(range(3))
results = [generate_with_thinking(x["instruction"]) for x in tqdm(sample)]
preds_sample = [r[0] for r in results]
finished = [r[1] for r in results]
print("закончила думать сама:", finished)

for p, x in zip(preds_sample, sample):
    print("вопрос:", x["inputs"]["question"])
    print("предсказание:", p)
    print("gold:", x["outputs"])
    print("exact match:", exact_match(p, x["outputs"]), ", F1:", round(f1(p, x["outputs"]), 2))
    print()

100%|██████████| 3/3 [01:37<00:00, 32.49s/it]

закончила думать сама: [False, False, False]
вопрос: В какую реку впадает река, притоком которой является Мурдюг?
предсказание: Вятка
gold: Каму;Камы
exact match: 0.0 , F1: 0.0

вопрос: Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?
предсказание: Глава государства
gold: президентов
exact match: 0.0 , F1: 0.0

вопрос: Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?
предсказание: Белуха
gold: Белуха;гора Белуха
exact match: 1.0 , F1: 1.0



Конечно, токенов для рассуждения не хватает. Но как будет видно дальше, результат всё равно улучшается, даже с учетом того, что нам приходится искуственно прерывать думание модели. 

In [80]:
results = [generate_with_thinking(x["instruction"], system_prompt) for x in tqdm(mera)]
predictions3 = [r[0] for r in results]

100%|██████████| 25/25 [11:15<00:00, 27.02s/it]


In [81]:
print("exact match:", sum(exact_match(p, x["outputs"]) for p, x in zip(predictions3, mera)) / len(mera))
print("F1:", sum(f1(p, x["outputs"]) for p, x in zip(predictions3, mera)) / len(mera))

for p, x in zip(predictions3, mera):
    print("вопрос:", x["inputs"]["question"])
    print("предсказание:", p)
    print("gold:", x["outputs"])
    print("exact match:", exact_match(p, x["outputs"]), ", F1:", round(f1(p, x["outputs"]), 2))
    print()

exact match: 0.12
F1: 0.30133333333333334
вопрос: В какую реку впадает река, притоком которой является Мурдюг?
предсказание: Вятка
gold: Каму;Камы
exact match: 0.0 , F1: 0.0

вопрос: Что за должность занимает глава государства, ставшего местом смерти Романа Николаевича Филипова?
предсказание: Глава государства
gold: президентов
exact match: 0.0 , F1: 0.0

вопрос: Какое место является наивысшей точкой горного хребта, к которому принадлежит Холзун?
предсказание: Белуха
gold: Белуха;гора Белуха
exact match: 1.0 , F1: 1.0

вопрос: В какую реку впадает река, притоком которой является Суходревка?
предсказание: Суходревка
gold: Шаня
exact match: 0.0 , F1: 0.0

вопрос: Как называется законодательный орган города, где родился Гор Шаваршевич Малакян?
предсказание: Ереванский городский совет
gold: ереванским городским советом
exact match: 0.0 , F1: 0.67

вопрос: Как называется река, питающаяся от реки, притоком которой является Ворговаш?
предсказание: Кисьма
gold: Сабля
exact match: 0.0 , F1: 0.0